# 01f — Old data (2023 Q1 – 2025 Q2): every branch joined to its main business

Stands alone: it reads its own 14 files and does not need the other preprocessing notebooks.

**Rules for this notebook**
- The raw files are never changed. Anything cleaned is saved as a new file.
- Nothing is dropped. Problem rows get a flag column instead.
- Data never goes into Git. Clear all outputs before committing.

**The preprocessing notebooks, in order** (each one loads what the one before saved):
1. `01a_id_check`: did the ID change, or only its column name? (stands alone)
2. `01b_all_quarters`: reads the 30 Excel files (2023 Q1 – 2026 Q3) once and saves `mc_all_quarters.parquet`
3. `01c_id_matching`: one ID per business, saves `mc_matched.parquet`
4. `01d_survival_label`: the one-year label, saves `labels.parquet`
5. `01e_ssic_classification`: SSIC from the name, saves `ssic_by_business.csv`, `business_names.csv`, `business_city.csv`
6. `01f_old_data_branches`: old data (2023 Q1 – 2025 Q2), every branch joined to its main (stands alone)

### Cell 1 — Setup
Finds the data. On Colab it downloads the raw files from the team's Google Drive folder (temporary copies,
gone when the session ends). On a laptop it reads the local copy in `DATA_DIR`.

The results that the next notebook loads go in `SAVED_DIR`: the folder **saved** inside the team's shared Drive
folder **Darb DataSets**. Everyone loads the same results, and they are still there in a new Colab session.
On Colab: add a shortcut to Darb DataSets in your My Drive first (Drive → right-click the folder → Organize →
Add shortcut → My Drive). Colab asks once to connect Drive.

In [ ]:
import os
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False

if IN_COLAB:
    !pip install -q gdown python-calamine
    DATA_DIR = Path('/content/darb-data')
    from google.colab import drive
    drive.mount('/content/drive')
    TEAM_DIR = Path('/content/drive/MyDrive/Darb DataSets')     # the team's shared folder (a shortcut in My Drive)
    if not TEAM_DIR.exists():
        raise FileNotFoundError('Darb DataSets is not in your My Drive: add a shortcut to it (see the text above), then run this cell again.')
    SAVED_DIR = TEAM_DIR / 'saved'                              # shared with the team, kept between Colab sessions
else:
    # local copy of the Drive folder "Darb DataSets" (outside Git and OneDrive)
    DATA_DIR = Path(os.environ.get('DARB_DATA', r'C:\Users\Nasser Alyahya\darb-data'))
    SAVED_DIR = DATA_DIR / 'saved'

MC_DIR = DATA_DIR / 'MC'
SAVED_DIR.mkdir(parents=True, exist_ok=True)
print('Running on Colab' if IN_COLAB else 'Running locally', '| data folder:', DATA_DIR, '| saved results:', SAVED_DIR)

### Cell 1b — Read an MC file (Excel only the first time)
Reading the big Excel files is the slow part. The first time a file is read, an **exact copy** of it (every row
and every column, all as text) is saved in `SAVED_DIR/mc_excel_copies`. After that the copy is read instead,
in seconds, and on Colab the Excel file is not downloaded again. Delete a copy to read its Excel file again.

In [ ]:
import pandas as pd

EXCEL_COPIES = SAVED_DIR / 'mc_excel_copies'
EXCEL_COPIES.mkdir(parents=True, exist_ok=True)

def has_copy(name):
    return (EXCEL_COPIES / f'{name}.parquet').exists()

def read_mc(name, usecols=None):
    # One MC file, all as text: from its saved copy if there is one, otherwise from Excel (and the copy is saved)
    copy = EXCEL_COPIES / f'{name}.parquet'
    if copy.exists():
        return pd.read_parquet(copy, columns=usecols)
    table = pd.read_excel(MC_DIR / name, engine='calamine', dtype=str)
    if all(isinstance(c, str) for c in table.columns):     # parquet needs text column titles
        temp = copy.with_suffix('.part')                     # a copy cut off halfway is never used
        table.to_parquet(temp, index=False)
        temp.rename(copy)
    return table[usecols] if usecols else table

Goal: **one row per business**, the main record (رئيسي) and all its branches (فرعي) together as one unit,
over all the old quarters: 2023 Q1, Q2, Q3, 2024 Q3, Q4, 2025 Q1, Q2 (2023 Q4, 2024 Q1 and 2024 Q2 are not in the Drive).

**Run after a restart:** Cell 1, then Cells 20 → 23.

### Cell 20 — Read the old quarters (only reads)
Reads the 14 files with all their columns, under the same column names as Cell 8. The 2023 files are new:
their e-commerce column is titled "لديه تجارة إلكترونية؟", and in **2023 Q3 PART1** the region values sit under
the e-commerce title, so that file's titles are corrected (the output shows a check).

In [ ]:
import gc
import pandas as pd

# The old quarters: (quarter, file name, Google Drive ID). 2023 Q4, 2024 Q1 and 2024 Q2 are not in the Drive.
OLD_FILES = [
    ('2023Q1', 'CR_2023-01q-active-crs_PART1.xlsx',   '1R-opbDgZDivW63kxt-WVe2G7x_zaDRym'),
    ('2023Q1', 'CR_2023-01q-active-crs_PART2.xlsx',   '1VgvjY6KwmqajrbGqOU9b95k9c5pFa18P'),
    ('2023Q2', 'CR_2023-02q-active-crs_PART1.xlsx',   '10rme8h_C8pPJUB-LHNfy2HlQUWHxLm0_'),
    ('2023Q2', 'CR_2023-02q-active-crs_PART2.xlsx',   '1_r4EBy4KJIwNgIJJnNPkMgQ3r_Vk4t9R'),
    ('2023Q3', 'CR_2023-03q-active-crs_PART1.xlsx',   '1lZ9kznVbx5eOgLSBDyGdOozQ-XoXmhbQ'),
    ('2023Q3', 'CR_2023-03q-active-crs_PART2.xlsx',   '1d03GvYu_035nXsmStTXoUVLm-4ybJukh'),
    ('2024Q3', 'CR_2024-03q-active-crs_PART1-2.xlsx', '1YAjKdTEzTf66Z_wK0nNVAboD_NoskbKV'),
    ('2024Q3', 'CR_2024-03q-active-crs_PART2-2.xlsx', '1bCd3HFyhCk9VbZgT79wxsIxgBBflKps0'),
    ('2024Q4', 'CR_2024-04q-active-crs_PART1.xlsx',   '1GPJUCZ5HiTa-_ZJBVLjrp58Ha53nOBGN'),
    ('2024Q4', 'CR_2024-04q-active-crs_PART2.xlsx',   '10y-NeQI6P78Sb_b3mqjTjcdpZq9zNvNK'),
    ('2025Q1', 'CR_2025-01q-active-crs_PART1.xlsx',   '1QTeifZpbmotbWmbZoliWoVcDxSK8y8uf'),
    ('2025Q1', 'CR_2025-01q-active-crs_PART2.xlsx',   '12LqZsJ8WHI4qUza_mXP4iUQPuwG9x_2d'),
    ('2025Q2', '2025-02q-active-crs_PART1-2.xlsx',    '1pAlWmlVTGwCY2XnwhuujvFz9w8ys3nfp'),
    ('2025Q2', '2025-02q-active-crs_PART2.xlsx',      '14LNdY7UCE-itRV4ubyfRzapGtwgoDda_'),
]
OLD_QUARTERS = ['2023Q1', '2023Q2', '2023Q3', '2024Q3', '2024Q4', '2025Q1', '2025Q2']

# One common name for every column (the same names as Cell 8, plus the 2023 e-commerce title)
OLD_NAMES = {
    'الرقم الوطني': 'id',
    'الرقم الرئيسي للسجل التجاري': 'main_id',       # for a branch: the ID of its main record
    'نوع السجل التجاري': 'record_type',
    'اسم السجل التجاري': 'name',
    'نوع الكيان': 'entity_type',
    'نوع النشاط التجاري (بالعربية)': 'legal_form',
    'رأس المال': 'capital',
    'اسم المدينة (بالعربية)': 'city',
    'اسم المنطقة (بالعربية)': 'region',
    'مفصح بالتجارة الإلكترونية؟': 'ecommerce',
    'مفصح بتجارة إلكترونية؟': 'ecommerce',
    'مفصح تجارة إلكترونية؟': 'ecommerce',
    'لديه تجارة إلكترونية؟': 'ecommerce',            # 2023
    'تاريخ الإصدار (ميلادي)': 'start_date',
}
# 2023 Q3 PART1 has no region title: its region values are under "مفصح بالتجارة الإلكترونية؟"
# and its e-commerce values under "لديه تجارة إلكترونية؟" (checked below)
FILE_FIXES = {'CR_2023-03q-active-crs_PART1.xlsx': {'مفصح بالتجارة الإلكترونية؟': 'region',
                                                    'لديه تجارة إلكترونية؟': 'ecommerce'}}
OLD_ORDER = ['quarter', 'source_file', 'id', 'main_id', 'record_type', 'name', 'entity_type', 'legal_form',
             'capital', 'region', 'city', 'ecommerce', 'start_date']
OLD_FEW = ['quarter', 'source_file', 'record_type', 'entity_type', 'legal_form', 'region', 'city', 'ecommerce']

if IN_COLAB:
    import gdown
    MC_DIR.mkdir(parents=True, exist_ok=True)
    for quarter, name, drive_id in OLD_FILES:
        if not (MC_DIR / name).exists() and not has_copy(name):
            print('downloading', name)
            gdown.download(id=drive_id, output=str(MC_DIR / name), quiet=True)

unknown_columns = set()
parts = []
for quarter, name, _ in OLD_FILES:
    part = read_mc(name)
    names = {**OLD_NAMES, **FILE_FIXES.get(name, {})}
    unknown_columns |= set(part.columns) - set(names)
    part = part.rename(columns=names)
    if part.columns.duplicated().any():
        print(f'  {name}: two columns got the same name {sorted(set(part.columns[part.columns.duplicated()]))}, the first is kept')
        part = part.loc[:, ~part.columns.duplicated()]
    for column in part.columns:
        part[column] = part[column].str.strip()
    part = part.assign(quarter=quarter, source_file=name).reindex(columns=OLD_ORDER)
    for column in OLD_ORDER:
        part[column] = part[column].astype('category' if column in OLD_FEW else 'string[pyarrow]')
    parts.append(part)

old = pd.concat(parts, ignore_index=True)
for column in OLD_FEW:
    old[column] = old[column].astype('category')
del parts
gc.collect()

import numpy as np
from IPython.display import display

def show(table, title):
    """Show a table with a title, numbers with thousands separators."""
    display(table.style.set_caption(title).format(lambda v: f'{v:,}' if isinstance(v, (int, np.integer)) else v))

files = old.groupby(['quarter', 'source_file'], observed=True).size().rename('rows').reset_index()
show(files, f'Rows per file: {len(old):,} rows in all | memory used: {old.memory_usage(deep=True).sum() / 1e9:.1f} GB')
print('Columns the notebook did not recognise:', sorted(unknown_columns) or 'none')
fixed = old[old['source_file'] == 'CR_2023-03q-active-crs_PART1.xlsx']
check = pd.DataFrame({'region (should be region names)': fixed['region'].value_counts().head(5).index.astype(str),
                      'rows ': fixed['region'].value_counts().head(5).to_numpy()})
show(check, 'Check of the 2023 Q3 PART1 title fix: region')
show(fixed['ecommerce'].value_counts().rename_axis('ecommerce (should be 0 / 1)').rename('rows').reset_index(),
     'Check of the 2023 Q3 PART1 title fix: e-commerce')


### Cell 21 — Clean the IDs and flag problem rows (adds one column, deletes nothing)
1. Every ID and main ID gets the same format: digits only (no spaces, no ".0").
2. `row_flag` for every row: `ok`, `empty_id`, `id_not_10_digits`, `header_row` (a column title read as data),
   `duplicate_in_quarter` (the 2nd copy of an ID in one quarter). Only `ok` rows are used next.

In [ ]:
import numpy as np
from IPython.display import display

def show(table, title):
    """Show a table with a title, numbers with thousands separators."""
    display(table.style.set_caption(title).format(lambda v: f'{v:,}' if isinstance(v, (int, np.integer)) else v))

# 1. The same format for every ID: digits only (no spaces, no ".0" left by Excel)
def clean_number(column):
    column = column.str.replace(r'\s+', '', regex=True).str.replace(r'\.0+$', '', regex=True)
    return column.mask(column == '')

old['id'] = clean_number(old['id'])
old['main_id'] = clean_number(old['main_id'])

# 2. A flag for every row. Nothing is deleted; only "ok" rows are used in the next cells.
TITLES = set(OLD_NAMES)                                    # a column title read as data = a header row
ROW_FLAGS = ['ok', 'empty_id', 'id_not_10_digits', 'header_row', 'duplicate_in_quarter']
id_is_10_digits = old['id'].str.fullmatch(r'\d{10}').fillna(False).to_numpy(dtype=bool)
codes = np.zeros(len(old), dtype=np.int8)
codes[~id_is_10_digits] = 2
codes[old['id'].isna().to_numpy()] = 1
codes[(old['id'].isin(TITLES) | old['name'].isin(TITLES)).to_numpy()] = 3
for quarter in OLD_QUARTERS:                               # the same ID twice in one quarter: the 2nd copy
    in_quarter = np.flatnonzero((old['quarter'] == quarter).to_numpy() & (codes == 0))
    codes[in_quarter[old['id'].iloc[in_quarter].duplicated().to_numpy()]] = 4
old['row_flag'] = pd.Categorical.from_codes(codes, ROW_FLAGS)
del codes
gc.collect()

per_quarter = pd.crosstab(old['quarter'], old['row_flag']).reindex(columns=ROW_FLAGS, fill_value=0)
per_quarter.insert(0, 'rows', old.groupby('quarter', observed=True).size())
per_quarter['main'] = old[old['record_type'] == 'رئيسي'].groupby('quarter', observed=True).size()
per_quarter['branch'] = old[old['record_type'] == 'فرعي'].groupby('quarter', observed=True).size()
per_quarter['type unknown'] = old[old['record_type'].isna()].groupby('quarter', observed=True).size()
per_quarter = per_quarter.fillna(0).astype(int)
per_quarter.index = per_quarter.index.astype(str)
per_quarter.loc['all quarters'] = per_quarter.sum()
show(per_quarter, 'Rows per quarter: row_flag and record type')


### Cell 22 — Link every branch to its main (adds two columns, deletes nothing)
A branch's `main_id` is the ID of its main record. It is looked up in the **same quarter** first, then in **any
old quarter** (a main can be missing from one incomplete file). A branch that points to another branch takes
that branch's main.

`link_case`: `main_record`, `linked_same_quarter`, `linked_other_quarter`, `linked_through_another_branch`,
`main_not_found`, `points_to_a_branch_with_no_main`, `main_id_not_10_digits`, `empty_main_id`, `not_used`.
`business_id`: the ID of the main record the row belongs to (a branch whose main is never found still belongs to
that main ID; a branch with no usable main ID stands alone under its own ID).

The match rate is shown per row and per different branch.

In [ ]:
use = (old['row_flag'] == 'ok').to_numpy()
is_main = use & (old['record_type'] == 'رئيسي').to_numpy()
is_branch = use & (old['record_type'] == 'فرعي').to_numpy()

# The main records of each quarter, and of all old quarters together
mains_in = {q: pd.Index(old.loc[is_main & (old['quarter'] == q).to_numpy(), 'id'].unique()) for q in OLD_QUARTERS}
all_mains = mains_in[OLD_QUARTERS[0]]
for q in OLD_QUARTERS[1:]:
    all_mains = all_mains.union(mains_in[q])
all_branches = pd.Index(old.loc[is_branch, 'id'].unique())
# For "a branch that points to another branch": the main of that other branch
main_of_branch = (old.loc[is_branch & old['main_id'].notna().to_numpy(), ['id', 'main_id']]
                  .drop_duplicates('id').set_index('id')['main_id'])

# Every branch row: find its main
b = old.loc[is_branch, ['quarter', 'id', 'main_id']]
main_id = b['main_id']
same_quarter = np.zeros(len(b), dtype=bool)
for q in OLD_QUARTERS:
    rows = (b['quarter'] == q).to_numpy()
    same_quarter[rows] = main_id[rows].isin(mains_in[q]).to_numpy()
other_quarter = main_id.isin(all_mains).to_numpy() & ~same_quarter
through = main_id.map(main_of_branch)
through_branch = through.isin(all_mains).to_numpy() & ~same_quarter & ~other_quarter
empty = main_id.isna().to_numpy()
not_10_digits = ~main_id.str.fullmatch(r'\d{10}').fillna(False).to_numpy(dtype=bool) & ~empty
points_to_branch = main_id.isin(all_branches).to_numpy()

CASES = ['main_record', 'linked_same_quarter', 'linked_other_quarter', 'linked_through_another_branch',
         'main_not_found', 'points_to_a_branch_with_no_main', 'main_id_not_10_digits', 'empty_main_id', 'not_used']
case = np.select([empty, same_quarter, other_quarter, through_branch, not_10_digits, points_to_branch],
                 [7, 1, 2, 3, 6, 5], default=4).astype(np.int8)

# business_id = the main record's ID. Linked branches take their main's ID.
# A branch whose main is not in these quarters still belongs to that main (main_not_found);
# a branch with no usable main_id stands alone, under its own ID.
business = np.where(np.isin(case, [1, 2, 4]), main_id.to_numpy(dtype=object),
           np.where(case == 3, through.to_numpy(dtype=object),
           np.where(case == 5, through.fillna(main_id).to_numpy(dtype=object), b['id'].to_numpy(dtype=object))))

codes = np.full(len(old), 8, dtype=np.int8)                 # not_used: flagged rows, unknown record type
codes[is_main] = 0
codes[is_branch] = case
old['link_case'] = pd.Categorical.from_codes(codes, CASES)
old['business_id'] = pd.Series(pd.NA, index=old.index, dtype='string[pyarrow]')
old.loc[is_main, 'business_id'] = old.loc[is_main, 'id']
old.loc[is_branch, 'business_id'] = business
del codes, b, main_id, through, business
gc.collect()

# Results
branch_rows = old[is_branch]
LINKED = ['linked_same_quarter', 'linked_other_quarter', 'linked_through_another_branch']
linked = branch_rows['link_case'].isin(LINKED)
per_branch = linked.groupby(branch_rows['id']).any()
show(pd.DataFrame({'count': [len(branch_rows), len(per_branch)],
                   'linked to a main': [f'{linked.mean():.2%}', f'{per_branch.mean():.2%}']},
                  index=['branch rows', 'different branches']), 'Match rate')

MEANING = {
    'linked_same_quarter': 'main found in the same quarter',
    'linked_other_quarter': 'main missing from that quarter, found in another old quarter',
    'linked_through_another_branch': 'main_id points to a branch; that branch\'s main was found',
    'main_not_found': 'the main\'s number is not a main record in any old quarter',
    'points_to_a_branch_with_no_main': 'main_id points to a branch whose main is also missing',
    'main_id_not_10_digits': 'main_id is not a 10-digit number',
    'empty_main_id': 'the branch has no main number',
}
cases = branch_rows['link_case'].value_counts().reindex(CASES[1:8]).rename('branch rows').to_frame()
cases['% of branch rows'] = (cases['branch rows'] / len(branch_rows)).map('{:.2%}'.format)
cases['linked?'] = ['yes' if c in LINKED else 'no' for c in cases.index]
cases['meaning'] = cases.index.map(MEANING)
show(cases, 'Every branch row by case')

per_q = pd.DataFrame({'branch rows': branch_rows.groupby('quarter', observed=True).size(),
                      'linked': linked.groupby(branch_rows['quarter'], observed=True).sum()})
per_q['not linked'] = per_q['branch rows'] - per_q['linked']
per_q['linked %'] = (per_q['linked'] / per_q['branch rows']).map('{:.2%}'.format)
show(per_q, 'Linked per quarter')

not_found = branch_rows[branch_rows['link_case'] == 'main_not_found']
show(not_found[['quarter', 'id', 'main_id', 'name', 'city']].sample(min(5, len(not_found)), random_state=1),
     'Examples: branches whose main was not found')
del branch_rows, not_found
gc.collect()


### Cell 22b — Where exactly are the branches with no main? (only reads)
Lists every branch row with `main_not_found`, `points_to_a_branch_with_no_main` or `empty_main_id`: its quarter,
file, **row number in the Excel file**, ID, main ID, name, city and start date. Counts per file, the first 10 of
each case, and all of them saved to `unlinked_branches.csv` (on the runtime's disk, never in Git).

Needs Cells 1, 20, 21 and 22 run first.

In [ ]:
# Where exactly are the branches that found no main? (only reads, plus one CSV file on the runtime's disk)
UNLINKED = ['main_not_found', 'points_to_a_branch_with_no_main', 'empty_main_id']

# The row number in the Excel file: rows keep their file order, row 1 is the title row
excel_row = old.groupby('source_file', observed=True).cumcount() + 2
where = old.loc[old['link_case'].isin(UNLINKED), ['quarter', 'source_file', 'id', 'main_id', 'name', 'city',
                                                   'start_date', 'link_case']]
where.insert(2, 'excel_row', excel_row[where.index])
where['start_date'] = where['start_date'].str[:10]
where['link_case'] = where['link_case'].cat.remove_unused_categories()

counts = pd.crosstab([where['quarter'], where['source_file']], where['link_case'])
counts['total'] = counts.sum(axis=1)
show(counts, f'Branch rows with no main: {len(where):,}, per file and case')
for case in UNLINKED:
    rows = where[where['link_case'] == case]
    show(rows.head(10).drop(columns='link_case'), f'{case}: {len(rows):,} rows (first 10)')

where.to_csv(DATA_DIR / 'unlinked_branches.csv', index=False, encoding='utf-8-sig')
print(f'All {len(where):,} rows saved to {DATA_DIR / "unlinked_branches.csv"}')


### Cell 22c — See the linking (only reads)
1. One real business: its main record and 10 of its branches, all with the same `business_id`.
2. Branches shown next to the name and city of the main they were linked to (4 random ones per linked case),
   to check by eye that each pair is the same business.

Needs Cells 1, 20, 21 and 22 run first.

In [ ]:
# See the linking with your own eyes (only reads)
LINKED_CASES = ['linked_same_quarter', 'linked_other_quarter', 'linked_through_another_branch']
COLS = ['quarter', 'id', 'record_type', 'main_id', 'name', 'city', 'business_id', 'link_case']

# 1. One real business: the business with the most branch rows. Its main record and 10 of its branches in 2025 Q2.
example_id = old.loc[old['link_case'].isin(LINKED_CASES), 'business_id'].value_counts().index[0]
one = old.loc[(old['business_id'] == example_id).fillna(False).to_numpy() & (old['quarter'] == '2025Q2').to_numpy(), COLS]
one = pd.concat([one[one['record_type'] == 'رئيسي'], one[one['record_type'] == 'فرعي'].head(10)])
show(one, f'One business (business_id {example_id}) in 2025 Q2: its main record and 10 of its branches. '
          'They all share the same business_id.')

# 2. Branches next to the main they were linked to: 4 per linked case, picked at random
main_rows = old.loc[old['link_case'] == 'main_record', ['id', 'name', 'city']].drop_duplicates('id', keep='last').set_index('id')
pairs = (old.loc[old['link_case'].isin(LINKED_CASES), ['quarter', 'link_case', 'id', 'name', 'city', 'business_id']]
         .groupby('link_case', observed=True, group_keys=False).apply(lambda g: g.sample(min(4, len(g)), random_state=3)))
pairs = pairs.rename(columns={'id': 'branch id', 'name': 'branch name', 'city': 'branch city', 'business_id': 'main id'})
pairs['main name'] = pairs['main id'].map(main_rows['name'])
pairs['main city'] = pairs['main id'].map(main_rows['city'])
show(pairs[['link_case', 'quarter', 'branch id', 'branch name', 'branch city', 'main id', 'main name', 'main city']],
     'Branches next to the main they were linked to (check that each pair looks like the same business)')
del one, main_rows, pairs
gc.collect()


### Cell 23 — One row per business (a new table `businesses`)
For every `business_id`: name, city, region, entity type, legal form and capital (from the latest main record),
start date (the main record's earliest), first and last quarter seen, quarters seen, and **`branch_count`**
(every different branch linked to it, counted once over all quarters).
`business_flag`: `has_main_record`, `main_never_seen` (only its branches are in the files),
`branch_with_no_main_id` (a branch standing alone).

In [ ]:
# One row per business: the main record and all its branches together, over all the old quarters
QUARTER_ORDER = {q: i for i, q in enumerate(OLD_QUARTERS)}
u = old.loc[old['business_id'].notna(), ['business_id', 'id', 'quarter', 'record_type', 'name', 'city', 'region',
                                         'entity_type', 'legal_form', 'capital', 'start_date']]
u = u.assign(order=u['quarter'].map(QUARTER_ORDER).astype(int), day=u['start_date'].str[:10])
is_own_main = ((u['record_type'] == 'رئيسي') & (u['id'] == u['business_id'])).fillna(False).astype(bool)

INFO = ['name', 'city', 'region', 'entity_type', 'legal_form', 'capital']
latest_row = u.sort_values('order').drop_duplicates('business_id', keep='last').set_index('business_id')
latest_main = u[is_own_main].sort_values('order').drop_duplicates('business_id', keep='last').set_index('business_id')

businesses = latest_row[INFO].copy()
has_main = businesses.index.isin(latest_main.index)
for column in INFO:                                     # the main record's details when there is one
    businesses.loc[has_main, column] = latest_main.loc[businesses.index[has_main], column].to_numpy()

# Start date: the main record's earliest start date (a branch's date only when the main is never seen)
start_main = u[is_own_main].groupby('business_id')['day'].min()
start_any = u.groupby('business_id')['day'].min()
businesses['start_date'] = start_main.reindex(businesses.index).fillna(start_any.reindex(businesses.index))

seen = u.groupby('business_id')['order'].agg(['min', 'max', 'nunique'])
businesses['first_quarter'] = seen['min'].map(dict(enumerate(OLD_QUARTERS)))
businesses['last_quarter'] = seen['max'].map(dict(enumerate(OLD_QUARTERS)))
businesses['quarters_seen'] = seen['nunique']

# Number of branches: every different branch ID linked to the business, counted once over all quarters
branches_of = u[((u['record_type'] == 'فرعي') & (u['id'] != u['business_id'])).fillna(False).astype(bool)]
businesses['branch_count'] = branches_of.groupby('business_id')['id'].nunique().reindex(businesses.index).fillna(0).astype(int)

orphan = old.loc[old['link_case'].isin(['empty_main_id', 'main_id_not_10_digits']), 'id'].unique()
businesses['business_flag'] = np.where(has_main, 'has_main_record',
                              np.where(businesses.index.isin(orphan), 'branch_with_no_main_id', 'main_never_seen'))
businesses['business_flag'] = businesses['business_flag'].astype('category')
businesses.index.name = 'business_id'
del u, latest_row, latest_main, start_main, start_any, seen, branches_of
gc.collect()

# Checks and results
show(pd.DataFrame({'value': [f'{len(businesses):,}', str(businesses.index.is_unique),
                             f'{int(businesses["branch_count"].sum()):,}']},
                  index=['businesses', 'one row each (no duplicates)', 'branches counted']), 'The businesses table')

FLAG_MEANING = {'has_main_record': 'the main record is in the data',
                'main_never_seen': 'only its branches are in the files (its main was not found)',
                'branch_with_no_main_id': 'a branch with no main number, standing alone'}
flags = businesses['business_flag'].value_counts().rename('businesses').to_frame()
flags['%'] = (flags['businesses'] / len(businesses)).map('{:.2%}'.format)
flags['meaning'] = flags.index.map(FLAG_MEANING)
show(flags, 'business_flag')

bins = pd.cut(businesses['branch_count'], [-1, 0, 1, 2, 5, 10, float('inf')], labels=['0', '1', '2', '3-5', '6-10', '11+'])
by_branches = bins.value_counts().sort_index().rename('businesses').rename_axis('number of branches').to_frame()
by_branches['%'] = (by_branches['businesses'] / len(businesses)).map('{:.2%}'.format)
show(by_branches, 'Businesses by number of branches')

show(businesses.nlargest(10, 'branch_count')[['name', 'city', 'branch_count', 'first_quarter', 'last_quarter']],
     'The 10 businesses with the most branches')
